In [1]:
!python --version

Python 3.11.13


재구성

In [2]:
!pip uninstall -y tf-keras keras-nightly keras==3.* tensorflow==2.16.* tensorflow==2.17.* tensorflow==2.18.* tf-nightly


Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0


In [3]:
!pip -q install "numpy==1.26.4" "ml-dtypes==0.2.0" "h5py==3.10.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 196.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 145.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.5.2 requires ml_dtypes>=0.4.0, but you have ml-dtypes 0.2.0 which is incompatible.
tensorstore 0.1.76 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.2.0 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [4]:
%env TF_USE_LEGACY_KERAS=1
!pip -q install "tensorflow==2.15.0.post1" "keras==2.15.0"


env: TF_USE_LEGACY_KERAS=1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 153.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 156.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 10.4 MB/s eta 0:00:00


In [5]:
!pip -q install --no-deps "deepctr==0.9.3"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 6.8 MB/s eta 0:00:00


재시작

In [1]:

import os, sys, types
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# 내부 경로에 Keras 2 심볼 매핑
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# init_ops_v2 대체 (DeepCTR가 참조)
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

print("TF:", tf.__version__, "| Keras:", keras.__version__, "→ shim ready")

TF: 2.15.0 | Keras: 2.15.0 → shim ready


In [2]:
import tensorflow as tf, keras, deepctr
print("TF:", tf.__version__)
print("Keras:", keras.__version__)
print("DeepCTR:", deepctr.__version__)

from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
from deepctr.models import DeepFM

print("DeepCTR import OK")


TF: 2.15.0
Keras: 2.15.0
DeepCTR: 0.9.3
DeepCTR import OK


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from deepctr.feature_column import SparseFeat, DenseFeat, VarLenSparseFeat, get_feature_names
from deepctr.models import DeepFM
import numpy as np


In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

train= pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/CRT/train_part1.parquet' , engine= 'pyarrow')


Mounted at /content/drive


In [5]:
df = train.copy()

In [6]:
df = df.drop(['l_feat_20' , 'l_feat_23','l_feat_2','l_feat_24'], axis = 1 )
df = df.drop(['history_a_1' , 'history_a_2' , 'history_a_3'], axis= 1)
df = df[~df['inventory_id'].isin([92, 21])]

    max_len : 800
    topk : 1000
    min_count = 500

In [7]:
# # padding
# # from tensorflow.keras.preprocessing.sequence import pad_sequences
# from keras.preprocessing.sequence import pad_sequences

# MAX_LEN = 150

# def parse_inc_to_array(s: str) -> np.ndarray:
#     # "1,2,3" -> [1,2,3] (int32), 음수 제외 +1 (0은 pad용으로 남김)
#     a = np.fromstring(str(s), sep=',', dtype=np.int32)
#     if a.size == 0:
#         return a
#     return a[a >= 0] + 1

# # seq → np.ndarray 리스트 (메모리에서만 보관)
# seq_arrs = df["seq"].map(parse_inc_to_array)

# # padding 배열 (df에 저장하지 않음)
# seq_padded = pad_sequences(
#     seq_arrs.tolist(),
#     maxlen=MAX_LEN,
#     dtype='int32',
#     padding='post',
#     truncating='pre',
#     value=0
# )

# # seq_len만 df에 저장 (필수)
# df["seq_len"] = seq_arrs.map(len).clip(upper=MAX_LEN).astype("int32")




In [7]:
import numpy as np
import pandas as pd

MAX_LEN = 150
PAD_ID = 0  # 0은 PAD, 실제 토큰은 +1 오프셋

def build_seq_padded_len(series: pd.Series, max_len: int, *, offset: int = 1, ignore_neg: bool = True):
    """
    series: 콤마 구분 문자열("9,18,269,...") 컬럼
    offset=1: 모델 전처리처럼 +1 오프셋(0은 PAD로 예약)
    return: seq_padded(int32, [N,max_len]), seq_len(int32, [N]), vocab_size(int)
    """
    # 미리 결과 배열을 한 번에 할당(메모리/속도 핵심)
    N = len(series)
    seq_padded = np.zeros((N, max_len), dtype=np.int32)
    seq_len    = np.zeros(N, dtype=np.int32)
    max_id     = 0

    # 판다스 오버헤드 줄이기: 바로 넘파이 배열로
    # astype(str)을 쓰면 NaN -> 'nan' 문자열이 되므로, 아래에서 비어 있으면 건너뜀
    vals = series.to_numpy(copy=False)

    for i in range(N):
        s = vals[i]
        if s is None or (isinstance(s, float) and np.isnan(s)):
            # 빈 시퀀스
            continue

        # 문자열로 캐스팅 (np.fromstring은 공백을 무시하므로 replace 불필요)
        text = s if isinstance(s, str) else str(s)
        if not text:
            continue

        # C 가속 파싱: 매우 빠름. 실패하면 size=0
        arr = np.fromstring(text, sep=',', dtype=np.int64)
        if arr.size == 0:
            continue

        if ignore_neg:
            # 음수 제거(있다면)
            arr = arr[arr >= 0]
            if arr.size == 0:
                continue

        if offset:
            # +1 오프셋 (0=PAD 유지)
            arr = arr + offset

        # 트렁케이팅: 최신 항목을 남기고 앞을 자름 (pre-truncating)
        L = arr.size
        if L > max_len:
            arr = arr[-max_len:]
            L = max_len

        # 패딩된 행에 앞쪽부터 복사 (post-padding)
        # arr는 int64이므로 복사 시 자동 캐스팅 → 비용 적음
        seq_padded[i, :L] = arr
        seq_len[i] = L

        # vocab_size 계산용 최대 id 갱신 (한 번에 끝)
        amax = int(arr.max()) if L > 0 else 0
        if amax > max_id:
            max_id = amax

    vocab_size = int(max_id + 1)  # PAD 포함
    return seq_padded, seq_len, vocab_size

# 사용 예시
seq_padded, seq_len, vocab_size = build_seq_padded_len(df["seq"], MAX_LEN, offset=1, ignore_neg=True)

# df에 길이만 저장(필요 시)
df["seq_len"] = seq_len


### 피처 열 생성


> DeepCTR에서 SparseFeat는 내부적으로 Embedding Lookup을 하기 때문에, 입력값은 반드시 0 ~ (vocabulary_size-1) 범위의 연속 정수 인덱스여야 한다.

>
    SparseFeat("inventory_id", vocabulary_size=16) 같이 정의하면, DeepCTR은 입력값을 0~15 정수 인덱스로 간주한다.

    그런데 실제 데이터는 2.0, 36.0, 37.0, …, 95.0처럼 흩어져 있음.

    이걸 그대로 Embedding lookup에 넣으면 → 인덱스 범위 초과 오류 또는 embedding index mismatch 발생.

> label encoding으로 맞춘다.


<br>


순서형의 경우 임베딩은 순서 정보를 기억하지 않는다.
>
    조회만 한다: e_k = Embedding[k] — k는 의미 있는 수가 아니라 “키”.

    순열 불변성: 첫 레이어가 ŷ = W·e_k + b일 때, 라벨을 임의로 섞고(순열 P) 임베딩과 가중치를 같이 섞으면 ŷ가 그대로.
    즉 모델은 라벨 순서에 무관하게 동치 해를 가짐.

    그래디언트 독립: 각 카테고리 벡터가 독립적으로 업데이트되어 연속성/단조성이 보장되지 않음. “2는 1과 3 사이”라는 규칙을 스스로 학습하리란 보장이 없다.

    그래서 임베딩으로 처리하면 ‘명목형처럼’ 취급되고, 순서(ordinal) 정보는 구조적으로 전달되지 않는다.

> 수치(연속/순서형) 특성은 보통 정규화해서 DenseFeat로 넣습니다.

In [8]:
label_feat = ['gender', 'age_group', 'day_of_week' , 'inventory_id']

encoders = {}

for feat in label_feat:
    le = LabelEncoder()
    df[feat] = le.fit_transform(df[feat])
    encoders[feat] = le

In [40]:
df['feat_a_13'].value_counts()

,count
feat_a_13,
0.000000,2967513
15.900000,190393
31.799999,40731
42.400002,4231
47.700001,190


In [41]:
HASH_BUCKET = 1_000_000
MAX_LEN = 150

EMBED_DIM = 8 # 임시로 통일

sparse_fixed = [
    SparseFeat('gender', vocabulary_size= df["gender"].nunique() , embedding_dim= EMBED_DIM , dtype = 'int32'),
    SparseFeat('age_group' , vocabulary_size= df["age_group"].nunique() , embedding_dim= EMBED_DIM , dtype = 'int32'),
    SparseFeat('day_of_week', vocabulary_size=df["day_of_week"].nunique() , embedding_dim= EMBED_DIM , dtype = 'int32'),
    SparseFeat('hour', vocabulary_size=24 , embedding_dim=EMBED_DIM, dtype = 'int32'),
    SparseFeat('inventory_id' , vocabulary_size= df["inventory_id"].nunique(), embedding_dim= EMBED_DIM, dtype = 'int32' ),

]

sparse_hash = [
    # SparseFeat('inventory_id' , vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , use_hash= True),
    SparseFeat('l_feat_14', vocabulary_size= HASH_BUCKET, embedding_dim= EMBED_DIM , use_hash= True ,dtype='string'),
]

# ordinal_sparse = [
#     SparseFeat('l_feat_3' ,vocabulary_size= 3 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('l_feat_27',vocabulary_size= 5 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_e_4' ,vocabulary_size= 4 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_1' ,vocabulary_size= 5 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_3' ,vocabulary_size= 6 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_4' ,vocabulary_size= 6 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_8' ,vocabulary_size= 7 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_13',vocabulary_size= 5 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_16',vocabulary_size= 7 , embedding_dim= EMBED_DIM, dtype = 'int32'),
#     SparseFeat('feat_a_18',vocabulary_size= 7 , embedding_dim= EMBED_DIM, dtype = 'int32')
# ]

# # ordinal feature는 denseFeat로 넣고 정규화
# ordinal_scores = [
#     DenseFeat('l_feat_3_ordscore', 1),
#     DenseFeat('feat_a_1_ordscore', 1),

# ]



varlen_seq  = VarLenSparseFeat(
    sparsefeat = SparseFeat('seq' ,
                            vocabulary_size= vocab_size , #2715931
                            embedding_dim= EMBED_DIM ,
                            # use_hash=True ,
                            dtype= 'int32', # 정수형 사용 -> 해시 안됨
                            ),
    maxlen = MAX_LEN,
    combiner = 'mean',
    length_name= 'seq_len',
    weight_name = None,
    weight_norm = False
)


seq_col = 'seq'
label_col = 'clicked'
seq_len_col = 'seq_len'

# dense feats 자동 수집
nominal_names = [f.name for f in (sparse_fixed + sparse_hash)]
# ordinal_names = [f.name for f in ordinal_sparse]

# exclude = set(nominal_names + ordinal_names + [label_col, seq_col, seq_len_col])
# dense_feats = [DenseFeat(c, 1) for c in df.columns if c not in exclude]


exclude = set(nominal_names  + [label_col, seq_col, seq_len_col])
dense_feats = [DenseFeat(c, 1) for c in df.columns if c not in exclude]



In [42]:
# 연속형 인코딩
dense_feat_names = [f.name for f in dense_feats]

mms = MinMaxScaler(feature_range=(0, 1))
df[dense_feat_names] = mms.fit_transform(df[dense_feat_names])


In [43]:
# linear_feature_columns = sparse_fixed + sparse_hash + ordinal_sparse + dense_feats
# dnn_feature_columns    = linear_feature_columns + [varlen_seq]

# feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


# 임시로 순서형 X
linear_feature_columns = sparse_fixed + sparse_hash  + dense_feats
dnn_feature_columns    = linear_feature_columns + [varlen_seq]

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)




# DIN: 고정길이 + 시퀀스(VarLenSparseFeat) 함께 사용

# dnn_feature_columns_din = linear_feature_columns + [varlen_seq]
# behavior_feature_list = ['inventory_id']  # query(현재 타깃)와 history 키 그룹 매칭


## 학습 샘플 생성 및 모델 학습

DeepCTR 모델은 내부적으로 특성 이름별로 Input Layer를 자동 생성한다.

그래서 입력을 dict 형태로 요구한다.


<br>
주의
>

    DataFrame의 seq는 건들지 말고, 모델에 넣을 때만 seq_padded/seq_len을 넘긴다


In [44]:
target = 'clicked'

train, test = train_test_split(df, test_size=0.2, random_state=2020)

train_model_input = {name: train[name] for name in feature_names}
test_model_input = {name: test[name] for name in feature_names}

train_y = train[target].to_numpy()
test_y = test[target].to_numpy()

In [45]:
target = 'clicked'

train, test = train_test_split(df, test_size=0.2, random_state=2020)

train_seq_padded, train_seq_len, _ = build_seq_padded_len(train["seq"], MAX_LEN, offset=1, ignore_neg=True)
test_seq_padded, test_seq_len, _ = build_seq_padded_len(test["seq"], MAX_LEN, offset=1, ignore_neg=True)


train_model_input = {}
test_model_input = {}

for name in feature_names:
    if name == 'seq':
        train_model_input[name] = train_seq_padded.astype('int32')
        test_model_input[name] = test_seq_padded.astype('int32')
    elif name == 'seq_len':
        train_model_input[name] = train_seq_len.astype('int32')
        test_model_input[name] = test_seq_len.astype('int32')

    elif name in [f.name for f in sparse_hash]:
        train_model_input[name] = train[name].values.astype(object)
        test_model_input[name] = test[name].values.astype(object)

    # elif name in [f.name for f in sparse_fixed + ordinal_sparse]:
    #     train_model_input[name] = train[name].astype('int32').values
    #     test_model_input[name] = test[name].astype('int32').values

    else:  # DenseFeat
        train_model_input[name] = train[name].astype('float32').values
        test_model_input[name] = test[name].astype('float32').values


train_y = train[target].to_numpy()
test_y = test[target].to_numpy()

In [46]:
model = DeepFM(linear_feature_columns, dnn_feature_columns, task='binary')
model.compile("adam", "binary_crossentropy",
              metrics=['binary_crossentropy'], )

In [47]:
# ============================================
# DeepCTR on Colab (Py 3.11) — TF 2.15 / Keras 2.15 호환 셋업
# ============================================
# 사용 시점:
# 1) 노트북 맨 위 셀에서 실행
# 2) DeepCTR, 모델 정의/fit 전에 반드시 실행
# --------------------------------------------
# 필요 시 설치(이미 설치되었다면 주석 유지하세요)
# !pip install -q "tensorflow==2.15.0" "keras==2.15.0" "deepctr==0.9.3"

import os, sys, types

# 레거시 tf.keras 사용 (Keras 3 경로로 빠지지 않도록)
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import keras

# ====== 출력(버전 확인) ======
print(f"TF: {tf.__version__} | Keras: {keras.__version__}")

# --------------------------------------------
# 1) DeepCTR가 기대하는 내부 경로 심볼 매핑 (tf.keras 내부 경로 → 외부 keras 심볼 프록시)
#    - tensorflow.python.keras.layers / initializers 를 keras.* 로 연결
sys.modules["tensorflow.python.keras.layers"] = keras.layers
sys.modules["tensorflow.python.keras.initializers"] = keras.initializers

# 2) DeepCTR가 참조하는 init_ops_v2 대체 모듈 주입
from keras.initializers import TruncatedNormal, Constant, glorot_uniform
init_v2 = types.ModuleType("tensorflow.python.ops.init_ops_v2")
init_v2.TruncatedNormal = TruncatedNormal
init_v2.Constant = Constant
init_v2.glorot_uniform = glorot_uniform
sys.modules["tensorflow.python.ops.init_ops_v2"] = init_v2

# 3) TF 2.15에 없는 내부 심볼 주입:
#    model.fit 경로에서 isinstance(ds, input_lib.DistributedDatasetInterface) 체크 시 AttributeError 방지
from tensorflow.python.distribute import input_lib as _input_lib
if not hasattr(_input_lib, "DistributedDatasetInterface"):
    class DistributedDatasetInterface:
        """Minimal shim for TF 2.15; acts as a marker interface only."""
        pass
    _input_lib.DistributedDatasetInterface = DistributedDatasetInterface

print("→ Internal shims installed OK")

# --------------------------------------------
# 4) DeepCTR import 및 버전 확인
try:
    import deepctr
    from deepctr.feature_column import SparseFeat, DenseFeat, get_feature_names
    from deepctr.models import DeepFM
    print(f"DeepCTR: {deepctr.__version__} (import OK)")
except Exception as e:
    print("DeepCTR import failed:", repr(e))
    raise

# (선택) 버전 가드 — 환경이 바뀌었을 때 빨리 눈치채기 위함
def _assert_versions(tf_req="2.15.", keras_req="2.15.", deepctr_req="0.9."):
    assert tf.__version__.startswith(tf_req), f"Require TF {tf_req}x, got {tf.__version__}"
    assert keras.__version__.startswith(keras_req), f"Require Keras {keras_req}x, got {keras.__version__}"
    assert deepctr.__version__.startswith(deepctr_req), f"Require DeepCTR {deepctr_req}x, got {deepctr.__version__}"

try:
    _assert_versions()
    print("→ Version guard passed (TF/Keras/DeepCTR expected range).")
except AssertionError as ae:
    print("! Version guard warning:", ae)

print("Shim setup complete. You can now build & fit DeepCTR models safely.")


TF: 2.15.0 | Keras: 2.15.0
→ Internal shims installed OK
DeepCTR: 0.9.3 (import OK)
→ Version guard passed (TF/Keras/DeepCTR expected range).
Shim setup complete. You can now build & fit DeepCTR models safely.


In [49]:
history = model.fit(train_model_input, train[target].values,
                    batch_size=256, epochs=10, verbose=2, validation_split=0.2, )
pred_ans = model.predict(test_model_input, batch_size=256)


print("test LogLoss", round(log_loss(test[target].values, pred_ans), 4))
print("test AUC", round(roc_auc_score(test[target].values, pred_ans), 4))

Epoch 1/10
8008/8008 - 189s - loss: 0.0872 - binary_crossentropy: 0.0872 - val_loss: 0.0892 - val_binary_crossentropy: 0.0892
Epoch 2/10
8008/8008 - 191s - loss: 0.0871 - binary_crossentropy: 0.0871 - val_loss: 0.0892 - val_binary_crossentropy: 0.0892
Epoch 3/10
8008/8008 - 193s - loss: 0.0869 - binary_crossentropy: 0.0869 - val_loss: 0.0899 - val_binary_crossentropy: 0.0899
Epoch 4/10
8008/8008 - 193s - loss: 0.0868 - binary_crossentropy: 0.0868 - val_loss: 0.0891 - val_binary_crossentropy: 0.0891
Epoch 5/10
8008/8008 - 192s - loss: 0.0867 - binary_crossentropy: 0.0867 - val_loss: 0.0894 - val_binary_crossentropy: 0.0894
Epoch 6/10
8008/8008 - 193s - loss: 0.0866 - binary_crossentropy: 0.0866 - val_loss: 0.0890 - val_binary_crossentropy: 0.0890
Epoch 7/10
8008/8008 - 194s - loss: 0.0865 - binary_crossentropy: 0.0865 - val_loss: 0.0909 - val_binary_crossentropy: 0.0909
Epoch 8/10
8008/8008 - 193s - loss: 0.0864 - binary_crossentropy: 0.0864 - val_loss: 0.0891 - val_binary_crossentropy:

롤백 or 훈련 종료 후 평가?

In [50]:
from sklearn.metrics import average_precision_score, log_loss
from sklearn.utils.class_weight import compute_class_weight

y_true = test[target].values.ravel().astype(np.int32)
y_pred = pred_ans.ravel().astype(float)

ap_pos = average_precision_score(y_true, y_pred)
ap_neg = average_precision_score(1 - y_true, 1 - y_pred)
ap_50  = 0.5 * (ap_pos + ap_neg)

w = compute_class_weight('balanced', classes=np.array([0,1]), y=y_true)
sw = np.where(y_true==0, w[0], w[1])
wll_50 = log_loss(y_true, y_pred, sample_weight=sw)

print(f"AP (50%): {ap_50:.4f}")
print(f"WLL(50%): {wll_50:.4f}")


AP (50%): 0.5296
WLL(50%): 1.8578
